# Minimal Transcript Cleaning

This notebook applies light cleaning to every transcript in `dataset/transcripts` and saves the cleaned versions into `dataset/cleaned`.

## Goal
The goal is to make the transcripts more consistent and easier to use for downstream modeling without removing too much linguistic information.

## What this notebook does
1. Loads all transcript `.txt` files from `dataset/transcripts`.
2. Applies a small set of normalization rules.
3. Saves the cleaned text files into `dataset/cleaned` using the same filenames.
4. Prints a simple before/after size summary.
5. Shows one example transcript before and after cleaning.

## Cleaning rules used here
- normalize unicode
- remove BOM characters
- flatten line breaks into a single text stream
- lowercase text for consistency
- remove obvious non-linguistic junk characters while keeping words, numbers, apostrophes, and common sentence punctuation
- normalize repeated spaces and punctuation spacing

## What this notebook does not do
It does not apply heavier NLP steps such as stopword removal, stemming, or lemmatization, because those can remove useful signal for classification.

In [1]:
# Import the utilities used for file handling and text cleaning.
from pathlib import Path
import re
import unicodedata

# Define the input and output folders.
# Input: raw transcripts created earlier in the pipeline.
# Output: cleaned transcripts with the same filenames.
ROOT = Path.cwd()
SOURCE_DIR = ROOT / 'dataset' / 'transcripts'
OUTPUT_DIR = ROOT / 'dataset' / 'cleaned'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load all transcript text files to be cleaned.
files = sorted(SOURCE_DIR.glob('*.txt'))
print(f'Found {len(files)} transcript files')
print(f'Output folder: {OUTPUT_DIR}')

Found 189 transcript files
Output folder: c:\Users\edgar\OneDrive\Documents\Term_8\Deep_learning\Project\phq8_depression_detection\dataset\cleaned


In [2]:
# Define the minimal cleaning function used for every transcript.
def clean_text(text: str) -> str:
    # Normalize unicode so visually similar characters are standardized.
    text = unicodedata.normalize('NFKC', text)

    # Remove BOM markers that sometimes appear in text files.
    text = text.replace('\ufeff', ' ')

    # Flatten transcript formatting into one normalized text stream.
    text = text.replace('\r', ' ').replace('\n', ' ')

    # Lowercase everything for consistency across files.
    text = text.lower()

    # Keep linguistic content and common sentence punctuation.
    # Anything outside this set is treated as likely noise and replaced with a space.
    text = re.sub(r"[^a-z0-9\s\.,!?;:'\-]", ' ', text)

    # Normalize apostrophes and repeated dashes that may appear in transcripts.
    text = text.replace("`", "'")
    text = re.sub(r"-{2,}", '-', text)

    # Standardize spacing and repeated punctuation without removing sentence structure.
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+([\.,!?;:])', r'\1', text)
    text = re.sub(r'([!?]){2,}', r'\1', text)
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r',{2,}', ',', text)

    # Remove leading and trailing whitespace from the final result.
    return text.strip()


In [3]:
# Apply the cleaning function to every transcript and save the result.
# The cleaned files keep the same filenames so they are easy to match with the originals.
for path in files:
    raw_text = path.read_text(encoding='utf-8', errors='ignore')
    cleaned_text = clean_text(raw_text)
    output_path = OUTPUT_DIR / path.name
    output_path.write_text(cleaned_text, encoding='utf-8')

print(f'Cleaned and saved {len(files)} files to {OUTPUT_DIR}')

Cleaned and saved 189 files to c:\Users\edgar\OneDrive\Documents\Term_8\Deep_learning\Project\phq8_depression_detection\dataset\cleaned


In [4]:
# Compare total text size before and after cleaning.
# This gives a quick sanity check that the cleaner ran and removed some noise.
total_chars_before = 0
total_chars_after = 0

for path in files:
    raw_text = path.read_text(encoding='utf-8', errors='ignore')
    cleaned_text = (OUTPUT_DIR / path.name).read_text(encoding='utf-8')
    total_chars_before += len(raw_text)
    total_chars_after += len(cleaned_text)

print(f'Total characters before cleaning: {total_chars_before:,}')
print(f'Total characters after cleaning:  {total_chars_after:,}')

Total characters before cleaning: 1,281,292
Total characters after cleaning:  1,280,807


In [5]:
# Show one example transcript before and after cleaning.
# This makes it easier to manually confirm that the cleaning is sensible.
sample_path = files[0]

print('Sample file:', sample_path.name)
print('\nOriginal:\n')
print(sample_path.read_text(encoding='utf-8', errors='ignore')[:600])

print('\nCleaned:\n')
print((OUTPUT_DIR / sample_path.name).read_text(encoding='utf-8')[:600])

Sample file: 300_P.txt

Original:

Good. Atlanta, Georgia. My parents are from here. I love it. Like the weather. I like the opportunities. Yes. It's a minute. I'm what easy. Ingestion. That's it. I took up business administration. Yeah. I am. In there. I'm on a break right now, but I plan on going back in the next semester. Probably to open up my own business. No. No specific reason. I just. Pretty. Once a year. Can you be a little bit more specific? No answer. I like reading both. I enjoy cooking. I'm not excited. It's great. Probably about two weeks ago. I press straight. I don't like bias. They don't. I'm what? I like to pl

Cleaned:

good. atlanta, georgia. my parents are from here. i love it. like the weather. i like the opportunities. yes. it's a minute. i'm what easy. ingestion. that's it. i took up business administration. yeah. i am. in there. i'm on a break right now, but i plan on going back in the next semester. probably to open up my own business. no. no specific reason. 